# Day 23 — Descriptive Statistics
### Statistics & Probability · Module 2 · Topics 2.1 – 2.3

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · 90 min concepts + 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | **Centre** — mean, median, mode, and when each lies | 25 min |
| 2 | **Spread** — range, IQR, variance, SD, and **ddof** | 35 min |
| 3 | **Shape** — skewness and kurtosis | 25 min |
| 4 | Mini build: a `describe()` that tells the truth | 5 min |
| 5 | **Practice notebook (separate file)** | 30 min |

---

## The question you have asked three times

- **Day 14:** "NumPy `std` uses `ddof=0`."
- **Day 15:** "pandas `std` uses `ddof=1`."
- **Day 19:** "state `ddof` explicitly."

Each time I said *we will come back to why*. **Today is that session**, and I am going to
show it by simulation rather than algebra.

## What changes in Module 2

In Module 1 your data **was** the thing you cared about: this spreadsheet, these 423 orders.

From today, your data is a **sample**, and you are trying to say something about a larger
world you cannot see. That one shift is the whole of statistics — and it is why the formulas
change.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(7)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.3 | numpy 1.26.4


In [ ]:
def f():
    try:
        return "A"
    except
print(f())

SyntaxError: invalid syntax (3871280542.py, line 4)

---
# 1. Centre

| Measure | Is | Best when |
|---|---|---|
| **Mean** | the balance point | roughly symmetric, no extremes |
| **Median** | the middle value | skewed, or outliers present |
| **Mode** | the most common value | categories, or repeated values |

The **mean is the only one that uses every value**. That is its strength and its weakness in
one sentence: it uses all your information, and one absurd value drags it anywhere.

The **median only cares about order**, so an extreme value counts for exactly one row —
however extreme it is.

In [ ]:
s = pd.Series([10, 12, 13, 14, 15, 16, 18])
print("mean  ", s.mean())
print("median", s.median())

# now replace the largest value with something absurd
s2 = pd.Series([10, 12, 13, 14, 15, 16, 9999])
print()
print("mean   after one bad value:", round(s2.mean(), 1))
print("median after one bad value:", s2.median())
print()
print("The mean moved by", round(s2.mean() - s.mean(), 1), "- the median by", s2.median() - s.median())

## 1.1 ⚠️ `mode()` returns a **Series**

In [ ]:
a = pd.Series([1, 2, 2, 3])
b = pd.Series([1, 1, 2, 2])        # a TIE

print("one mode  :", a.mode().tolist())
print("two modes :", b.mode().tolist())
print()
print("b.mode()[0] gives", b.mode()[0], "- which silently hides the tie")
print("so check first:", len(b.mode()), "modes")

# mean() and median() return a number; mode() cannot, because ties exist.

## 1.2 When the mean describes nobody

In [ ]:
# 1,000 salaries: 950 ordinary staff, 50 executives
inc = np.r_[rng.normal(45000, 8000, 950), rng.normal(400000, 60000, 50)]
sal = pd.Series(inc)

print(f"mean   {sal.mean():>10,.0f}")
print(f"median {sal.median():>10,.0f}")
print(f"gap    {sal.mean() - sal.median():>10,.0f}")
print()
print(f"people earning BELOW the average: {(sal < sal.mean()).mean() * 100:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sal, bins=60, color="#3E7CB1", edgecolor="white")
ax.axvline(sal.median(), color="#2FA98C", linestyle="--", linewidth=2,
           label=f"median {sal.median():,.0f}")
ax.axvline(sal.mean(), color="#E8674C", linestyle="--", linewidth=2,
           label=f"mean {sal.mean():,.0f}")
ax.set_title("1,000 salaries: the mean sits where almost nobody is")
ax.set_xlabel("Salary (Rs)"); ax.set_ylabel("People")
ax.legend()
plt.show()

> **The rule.** When the mean and the median differ substantially, **report both** — the gap
> between them *is* the finding.
>
> This is Day 21's "average salary" case again, and it is why every serious income statistic
> you will ever read is a median.

---
# 2. Spread

| Measure | Definition | Robust? | Units |
|---|---|---|---|
| Range | `max - min` | No — uses only the extremes | same as data |
| IQR | `Q3 - Q1`, the middle 50% | **Yes** — ignores the tails | same as data |
| Variance | mean squared distance from the mean | No | **squared** |
| Std deviation | the square root of the variance | No | same as data |

## 2.1 Why square at all?

The obvious measure — the average distance from the mean — is **always exactly zero**,
because the positives and negatives cancel. That is what "balance point" means.

In [ ]:
x = pd.Series([2, 4, 4, 4, 5, 5, 7, 9])
dev = x - x.mean()

print("deviations    :", dev.tolist())
print("their sum     :", round(dev.sum(), 10), "   <- always zero")
print("their mean    :", round(dev.mean(), 10), "   <- so this is useless")
print()
print("mean of SQUARED deviations:", round((dev ** 2).mean(), 3), "  = the variance")
print("square root of that       :", round(np.sqrt((dev ** 2).mean()), 3), "  = the SD")
print()
print("check against numpy       :", round(x.std(ddof=0), 3))

Squaring removes the signs. The square root then puts you **back into the original units**,
which is why we quote the SD and not the variance.

> You cannot interpret "a variance of 4.2 million rupees **squared**". You can interpret
> "a standard deviation of 2,049 rupees".

---
# 2.2 ⚠️ Why divide by n − 1? Here is the proof.

Instead of deriving it, let us **measure** it. Build a population we know the truth about,
take 200,000 small samples from it, and see which formula gets the right answer.

In [ ]:
# A dedicated generator, so this simulation gives the same numbers every time
# no matter which cells you ran before it.
sim = np.random.default_rng(2026)

POP = sim.normal(100, 15, 1_000_000)
true_var = POP.var()                      # the truth, because we have everyone
print(f"true population variance: {true_var:.2f}")

n, trials = 5, 500_000
samples = sim.choice(POP, size=(trials, n))

v0 = samples.var(axis=1, ddof=0).mean()   # divide by n
v1 = samples.var(axis=1, ddof=1).mean()   # divide by n - 1

print()
print(f"average sample variance, /n      (ddof=0): {v0:7.2f}   bias {v0 - true_var:+6.2f}")
print(f"average sample variance, /(n-1)  (ddof=1): {v1:7.2f}   bias {v1 - true_var:+6.2f}")

**Dividing by `n` undershoots the truth — not occasionally, but on average, every time.**

And the size of the miss is not arbitrary. Compare the shortfall with `(n−1)/n`:

In [ ]:
print(f"observed  v0 / true  = {v0 / true_var:.4f}")
print(f"predicted (n-1) / n  = {(n - 1) / n:.4f}")

In [ ]:
# The real evidence: the shortfall tracks (n-1)/n at EVERY sample size.
print(f"{'n':>4} {'/n gives':>10} {'/(n-1) gives':>13} {'observed':>10} {'predicted':>10}")
print(f"{'':>4} {'':>10} {'':>13} {'ratio':>10} {'(n-1)/n':>10}")
for n in [3, 5, 10, 50, 200]:
    sm = sim.choice(POP, size=(200_000, n))
    a = sm.var(axis=1, ddof=0).mean()
    b = sm.var(axis=1, ddof=1).mean()
    print(f"{n:>4} {a:>10.2f} {b:>13.2f} {a/true_var:>10.3f} {(n-1)/n:>10.3f}")

The observed ratio tracks `(n−1)/n` at every sample size — and notice how the `/n` column
climbs towards the truth as `n` grows while the `/(n−1)` column is already there. **That is
why the correction matters most on small samples, and why nobody notices it on large ones.**

### The intuition, in one sentence

You measured each value's distance from the **sample** mean — but the sample mean was
calculated *from those same values*, so it sits closer to them than the true population mean
does. Your distances are therefore **too small, always**.

Dividing by the smaller number `n − 1` inflates the answer by exactly the right amount to
compensate.

## 2.3 So which one do you use?

| | Use when your data… | Examples |
|---|---|---|
| **`ddof=0`** *(population)* | **is** everything you care about | all 60 students in this class; every order in 2025 |
| **`ddof=1`** *(sample)* | stands in for something bigger | 200 customers → all customers; a month of readings → the sensor |

The second is almost always the case, **which is why pandas made it the default**.

In [ ]:
x = np.array([88., 71., 64., 91., 55.])

print("numpy  np.std(x)          :", round(np.std(x), 4), "  (ddof=0)")
print("pandas pd.Series(x).std() :", round(pd.Series(x).std(), 4), "  (ddof=1)")
print()
print("same data, different answers - and now you know why neither is wrong.")
print()
print("state it explicitly whenever the number will be reported:")
print("  np.std(x, ddof=1)        :", round(np.std(x, ddof=1), 4))
print("  pd.Series(x).std(ddof=0) :", round(pd.Series(x).std(ddof=0), 4))

---
# 3. Shape

## 3.1 Same mean. Similar SD. Completely different data.

In [ ]:
A_ = rng.normal(50, 10, 100_000)                                   # one hump
B_ = np.r_[rng.normal(40, 4, 50_000), rng.normal(60, 4, 50_000)]   # two humps

for name, d in [("A (one hump) ", A_), ("B (two humps)", B_)]:
    s_ = pd.Series(d)
    print(f"{name}  mean {s_.mean():5.2f}   sd {s_.std():5.2f}   "
          f"skew {s_.skew():+5.2f}   kurtosis {s_.kurt():+6.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
axes[0].hist(A_, bins=60, color="#3E7CB1", edgecolor="white")
axes[0].set_title("A — mean 50, sd 10")
axes[1].hist(B_, bins=60, color="#E8674C", edgecolor="white")
axes[1].set_title("B — mean 50, sd 11")
for ax in axes:
    ax.axvline(50, color="#152238", linestyle="--", linewidth=2)
    ax.set_xlabel("value")
axes[0].set_ylabel("count")
fig.suptitle('Both have a mean of 50. In B, almost nobody IS 50.', y=1.04)
fig.tight_layout()
plt.show()

**The mean of B describes a person who does not exist.** Report "mean 50, SD 10" for either
and the reader pictures A.

Day 20's Anscombe quartet made this point for *relationships*. This is the same point for
*distributions*: two numbers cannot describe a shape.

## 3.2 Skewness and kurtosis

| `skew()` | Means | Example |
|---|---|---|
| `> 0` | long tail to the **right** | income, delays |
| `≈ 0` | symmetric | heights |
| `< 0` | long tail to the **left** | exam marks with a ceiling |

| `kurt()` | Means |
|---|---|
| `> 0` | **heavy** tails — more extremes than a normal curve |
| `≈ 0` | normal-like tails |
| `< 0` | **light** tails — flat, or two humps |

> pandas reports **excess** kurtosis, so a normal curve scores 0 rather than 3. Other tools
> use the other convention — check before comparing.

In [ ]:
examples = {
    "salaries (right tail)": sal,
    "heights (symmetric)  ": pd.Series(rng.normal(170, 10, 10_000)),
    "marks w/ ceiling     ": pd.Series(100 - rng.gamma(2, 6, 10_000)).clip(0, 100),
    "two humps            ": pd.Series(B_),
}
for name, d in examples.items():
    print(f"{name}  skew {d.skew():+6.2f}   kurtosis {d.kurt():+7.2f}")

**You rarely quote these numbers to a client.** You use them to decide *which summary to
quote*: a skew of 4.28 tells you the mean will mislead before you have plotted anything.

## 3.3 Choosing the honest summary

| If the data is… | Centre | Spread | Because |
|---|---|---|---|
| Roughly symmetric | mean | std | every value contributes, nothing distorts |
| Skewed | **median** | **IQR** | the mean chases the tail |
| Full of outliers | **median** | **IQR** | quartiles ignore the extremes |
| Bimodal | neither — **split it** | — | one summary describes no one |
| Categorical | mode | counts | there is no arithmetic to do |

**And always report `n`.**

---
# 4. Mini build — a `describe()` that tells the truth

In [ ]:
def honest_summary(s, name="value"):
    """Centre, spread and shape - plus a recommendation describe() will not give you."""
    missing = int(s.isna().sum())
    s = s.dropna()
    skew = s.skew()

    out = {
        "n":        len(s),
        "missing":  missing,
        "mean":     round(s.mean(), 2),
        "median":   round(s.median(), 2),
        "std":      round(s.std(), 2),            # ddof=1, sample
        "std_pop":  round(s.std(ddof=0), 2),      # ddof=0, population
        "IQR":      round(s.quantile(.75) - s.quantile(.25), 2),
        "skew":     round(skew, 2),
        "kurtosis": round(s.kurt(), 2),
    }

    gap = abs(out["mean"] - out["median"]) / out["std"] if out["std"] else 0
    out["report"] = ("MEDIAN and IQR - the data is skewed"
                     if abs(skew) > 1 or gap > 0.3
                     else "mean and std - roughly symmetric")
    return pd.Series(out, name=name)

In [ ]:
pd.concat([
    honest_summary(sal, "salaries"),
    honest_summary(pd.Series(rng.normal(170, 10, 1000)), "heights"),
], axis=1)

`describe()` gives you numbers. This gives you a **decision** — which is what you were
actually after.

Keep this function; it reappears in Module 2's project.

---
# 5. Recap — the twelve things to remember

1. Describe any distribution with **centre, spread and shape**.
2. The mean uses every value — its strength and its weakness.
3. The median only cares about order, so outliers count once.
4. Mean far from median = skew. **Report both**; the gap is the finding.
5. In the salary example, **94% of people earn below the average**.
6. `mode()` returns a **Series**, because ties are possible.
7. Variance is in squared units; quote the **standard deviation**.
8. ⚠️ Dividing by `n` understates the truth by exactly **(n−1)/n**.
9. So **`ddof=1` for a sample**, **`ddof=0` for a whole population**.
10. NumPy defaults to 0, pandas to 1. Neither is wrong — **state it**.
11. Skew says *which tail*; kurtosis says *how heavy*. Both guide the choice.
12. Two humps can share a mean and SD with one hump. **Always plot.**

---

### 📝 Now open **`Day23_Practice_Questions.ipynb`**.

### Homework
- Reproduce the n−1 simulation yourself and try n = 3, 10 and 50.
- Find a real dataset where the mean and median disagree, and explain why.
- Run `honest_summary` on three columns of your Day 21 project data.

### Next class — Topic 2.4: The normal distribution
The bell curve, the 68–95–99.7 rule, and z-scores — where today's standard deviation stops
being a description and becomes a **ruler you can measure anything with**.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*